# Modulo Catalogo — Filtri e scoperta

Secondo blocco del catalogo, dopo la ricerca base. Aggiunge:

1. **Più votati in assoluto** (discovery della home)
2. **Filtro per genere** con soglia voti (versione corretta)
3. **Filtro per anno** o intervallo
4. **Ricerca combinata** (genere + anno + voto minimo in un solo filtro)

Tutte le classifiche usano una **soglia minima di voti**: ordinare per sola media premia i film con pochissimi voti, quindi si tengono solo quelli con abbastanza valutazioni da rendere la media affidabile.

In [1]:
from connessione import get_db

db = get_db()
film = db["film"]

# soglia minima di voti per le classifiche (tarata a 1000 sul catalogo)
SOGLIA_VOTI = 1000

# campi restituiti nelle liste (documento leggero, non la scheda intera)
PROIEZIONE = {"titolo": 1, "anno_uscita": 1, "generi": 1,
              "rating.tmdb_media": 1, "rating.tmdb_voti": 1}

## 1. Più votati in assoluto

Filtro: `{ "rating.tmdb_voti": { "$gte": 1000 } }`, ordinati per `rating.tmdb_media` decrescente.

Serve la home del catalogo. L'ordinamento sfrutta l'indice `idx_rating`; la soglia sui voti tiene fuori i film con medie gonfiate da poche valutazioni.

In [2]:
def film_piu_votati(limite=20):
    """I migliori film del catalogo, con almeno SOGLIA_VOTI voti."""
    filtro = {"rating.tmdb_voti": {"$gte": SOGLIA_VOTI}}
    return list(
        film.find(filtro, PROIEZIONE)
            .sort("rating.tmdb_media", -1)
            .limit(limite)
    )

In [3]:
for f in film_piu_votati(10):
    print(f"{f['rating']['tmdb_media']:.1f}  ({f['rating']['tmdb_voti']:>6} voti)  {f['titolo']} ({f.get('anno_uscita')})")

8.9  (  1875 voti)  Swapped - Al tuo posto (2026)
8.7  ( 30702 voti)  Le ali della libertà (1994)
8.7  (  3328 voti)  Michael (2026)
8.7  ( 23120 voti)  Il padrino (1972)
8.7  (  5759 voti)  L'ultima missione: Project Hail Mary (2026)
8.6  ( 14018 voti)  Il padrino - Parte II (1974)
8.6  ( 17577 voti)  Schindler's List (1993)
8.6  ( 10096 voti)  La parola ai giurati (1957)
8.5  ( 18495 voti)  La città incantata (2001)
8.5  ( 36028 voti)  Il cavaliere oscuro (2008)


## 2. Filtro per genere (con soglia voti)

Versione corretta rispetto alla ricerca base: la soglia sui voti elimina i titoli sconosciuti con media altissima ma pochi voti.

Filtro: `{ "generi": "Azione", "rating.tmdb_voti": { "$gte": 1000 } }`. L'equality su `generi` più l'ordinamento per voto sono serviti dall'indice composto `idx_genere_rating`.

In [4]:
def film_per_genere(genere, limite=20):
    """Film di un genere, dal più votato, con almeno SOGLIA_VOTI voti."""
    filtro = {"generi": genere, "rating.tmdb_voti": {"$gte": SOGLIA_VOTI}}
    return list(
        film.find(filtro, PROIEZIONE)
            .sort("rating.tmdb_media", -1)
            .limit(limite)
    )

In [10]:
for f in film_per_genere("Horror", 10):
    print(f"{f['rating']['tmdb_media']:.1f}  {f['titolo']} ({f.get('anno_uscita')})")

8.4  Psyco (1960)
8.3  Obsession (2026)
8.2  Shining (1980)
8.2  Alien (1979)
8.1  La cosa (1982)
8.0  Lee Cronin - La Mummia (2026)
7.8  Train to Busan (2016)
7.7  L'esorcista (1973)
7.7  Il cigno nero (2010)
7.7  Lo squalo (1975)


## 3. Filtro per anno o intervallo

Filtro con gli operatori di confronto `$gte` (da) e `$lte` (fino a) su `anno_uscita`.
Esempi: dal 2010 in poi `{ "anno_uscita": { "$gte": 2010 } }`; anni '90 `{ "anno_uscita": { "$gte": 1990, "$lte": 1999 } }`.
Usa l'indice `idx_anno`.

In [6]:
def film_per_anni(anno_min=None, anno_max=None, limite=20):
    """Film in un intervallo di anni, dal più recente. Estremi opzionali."""
    intervallo = {}
    if anno_min is not None:
        intervallo["$gte"] = anno_min
    if anno_max is not None:
        intervallo["$lte"] = anno_max
    filtro = {"anno_uscita": intervallo} if intervallo else {}
    return list(
        film.find(filtro, PROIEZIONE)
            .sort("anno_uscita", -1)
            .limit(limite)
    )

In [7]:
# esempio: film degli anni '90
for f in film_per_anni(1990, 1999, 10):
    print(f"{f.get('anno_uscita')}  {f['titolo']}")

1999  Arlington Road - L'inganno
1999  Il mistero di Sleepy Hollow
1999  Notting Hill
1999  Ragazze interrotte
1999  Gioco a due
1999  Giorni contati - End of Days
1999  Echi mortali
1999  Stuart Little - Un topolino in gamba
1999  Sexo, pudor y lágrimas
1999  Happy end


## 4. Ricerca combinata

La funzione che alimenta la pagina catalogo: mette insieme genere, intervallo di anni e voto minimo in **un solo filtro**. Le condizioni si sommano (AND logico) perché sono chiavi diverse dello stesso documento-filtro. Tutti i parametri sono opzionali: se non passi nulla, ottieni i più votati.

In [8]:
def cerca_film(genere=None, anno_min=None, anno_max=None,
               voti_min=SOGLIA_VOTI, ordina_per="rating.tmdb_media",
               decrescente=True, limite=20):
    """Ricerca del catalogo con filtri combinati opzionali."""
    filtro = {}
    if genere:
        filtro["generi"] = genere
    intervallo = {}
    if anno_min is not None:
        intervallo["$gte"] = anno_min
    if anno_max is not None:
        intervallo["$lte"] = anno_max
    if intervallo:
        filtro["anno_uscita"] = intervallo
    if voti_min:
        filtro["rating.tmdb_voti"] = {"$gte": voti_min}

    direzione = -1 if decrescente else 1
    return list(
        film.find(filtro, PROIEZIONE)
            .sort(ordina_per, direzione)
            .limit(limite)
    )

In [9]:
# esempio: i migliori film d'azione dal 2010 in poi
for f in cerca_film(genere="Azione", anno_min=2010, limite=10):
    print(f"{f['rating']['tmdb_media']:.1f}  {f['titolo']} ({f.get('anno_uscita')})")

8.4  Spider-Man - Un nuovo universo (2018)
8.4  Inception (2010)
8.3  Spider-Man: Across the Spider-Verse (2023)
8.3  The Punisher: One Last Kill (2026)
8.2  Avengers: Infinity War (2018)
8.2  Avengers: Endgame (2019)
8.2  Demon Slayer - Il treno Mugen (2020)
8.2  Justice League Dark: Apokolips War (2020)
8.2  Top Gun: Maverick (2022)
8.1  Jujutsu Kaisen 0 - Il film (2021)


### Come si costruisce il filtro combinato

`cerca_film` mostra un pattern utile: il filtro è un dizionario che si **riempie a pezzi** a seconda dei parametri passati. Ogni chiave aggiunta è una condizione in più che si somma alle altre. È lo stesso principio con cui una pagina con menu a tendina compone la ricerca: l'utente sceglie genere e anni, e il filtro cresce di conseguenza. Questa è la funzione da collegare alla pagina Streamlit del catalogo.